# Iniciando o Spark

In [1]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_atraso") \
    .getOrCreate()

# Importando bibliotecas

In [4]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [5]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

#Alterar o path_padrao caso seus arquivos não estejam nesse mesmo caminho
path_padrao = "/content/gdrive/Othercomputers/Meu laptop"

# Buckets e nomes de saída
bucket_base = "base_atraso"
bucket_raw = f"{path_padrao}/Database_raw/book_atraso"
bucket_trusted = f"{path_padrao}/Database_trusted/book_atraso"
bucket_control = f"{path_padrao}/Database_control/book_atraso"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_raw:", bucket_raw)
print("bucket_trusted:", bucket_trusted)
print("bucket_control:", bucket_control)


PROCESS_DATE: 20260210
REF_PERIOD: 202602
dthproc: 20260210102643
bucket_raw: /content/gdrive/Othercomputers/Meu laptop/Database_raw/book_atraso
bucket_trusted: /content/gdrive/Othercomputers/Meu laptop/Database_trusted/book_atraso
bucket_control: /content/gdrive/Othercomputers/Meu laptop/Database_control/book_atraso


#  Leitura da camada Raw

In [6]:
path_raw = bucket_raw
parquet_files = os.path.join(bucket_raw, "dados_faturamento")
#print("Raw path:", path_files)

parquet_files = [parquet_files for f in os.listdir(parquet_files) if f.endswith('.parquet')]
df_raw_atraso= spark.read.parquet(*parquet_files, header=True, inferSchema=True)
df_raw_atraso.createOrReplaceTempView("raw_base_atraso")

print(log(), "Registros na Raw:", df_raw_atraso.count())
#df_raw_atraso.printSchema()

2026-02-10 13:26:59 >>> Registros na Raw: 316113160


In [7]:
#csv_files = [path_raw for f in os.listdir(bucket_raw) if f.endswith('.csv')]
#df_dim_atraso = spark.read.csv(*csv_files, header=True, inferSchema=True, sep=',')

csv_files = os.path.join(path_raw, "BI_DIM_TIPO_FATURAMENTO.csv")
df_dim_atraso = spark.read.csv(csv_files, header=True, inferSchema=True, sep=',')

df_dim_atraso.createOrReplaceTempView("df_tipo_faturamento")

print(log(), "Registros na Raw:", df_dim_atraso.count())
df_dim_atraso.show(5, truncate=False)

2026-02-10 13:27:35 >>> Registros na Raw: 72
+-------------------+--------------------+--------------------+----------------+------------------+--------------------------+
|DW_TIPO_FATURAMENTO|DSC_TIPO_FATURAMENTO|COD_TIPO_FATURAMENTO|DAT_EXPIRACAO_DW|DAT_CRIACAO_DW    |DSC_TIPO_FATURAMENTO_ABREV|
+-------------------+--------------------+--------------------+----------------+------------------+--------------------------+
|32527              |Refund              |R                   |NULL            |28MAR2017:08:00:40|REFUND                    |
|32528              |Credit              |C                   |NULL            |28MAR2017:08:00:40|CREDIT                    |
|32529              |Claro               |B                   |NULL            |28MAR2017:08:00:40|SERVICE                   |
|32530              |Service Deposit     |D                   |NULL            |28MAR2017:08:00:40|DEPOSIT                   |
|32531              |Reversal            |RV                  |NUL

# Processamento tipagem para camada Trusted

In [8]:
df_trusted_atraso = spark.sql(f"""
SELECT
    '{dthproc}' AS ts_proc,
    '{dthproc}' AS ts_proc_partition,
    CAST(NUM_CPF AS STRING)                                     AS NUM_CPF,
    TO_DATE(DAT_REFERENCIA, 'ddMMMyyyy:HH:mm:ss')               AS DAT_REFERENCIA,
    CAST(NUM_FATURA_HASH AS STRING)                              AS NUM_FATURA_HASH,
    CAST(NUM_ENT_SEQ_FATURA AS INT)                              AS NUM_ENT_SEQ_FATURA,
    CAST(CONTRATO AS BIGINT)                                     AS CONTRATO,
    CAST(DW_UN_NEGOCIO AS INT)                                   AS DW_UN_NEGOCIO,
    CAST(DW_HIS_PONTO_VENDA_COMTA AS BIGINT)                     AS DW_HIS_PONTO_VENDA_COMTA,
    CAST(DW_NUM_CLIENTE AS BIGINT)                               AS DW_NUM_CLIENTE,
    CAST(DW_AREA AS INT)                                         AS DW_AREA,
    CAST(DW_CICLO AS INT)                                        AS DW_CICLO,
    CAST(DW_TIPO_CLIENTE_CONTA AS INT)                           AS DW_TIPO_CLIENTE_CONTA,
    CAST(DW_OFERTA AS INT)                                       AS DW_OFERTA,
    CAST(DW_FAIXA_AGING_FATURA AS INT)                           AS DW_FAIXA_AGING_FATURA,
    CAST(DW_FAIXA_AGING_DIVIDA AS INT)                           AS DW_FAIXA_AGING_DIVIDA,
    CAST(DW_FAIXA_TEMPO_BASE AS INT)                             AS DW_FAIXA_TEMPO_BASE,
    CAST(DW_FAIXA_AGING_PROX_FECH AS INT)                        AS DW_FAIXA_AGING_PROX_FECH,
    CAST(DW_TIPO_FATURAMENTO AS INT)                             AS DW_TIPO_FATURAMENTO,
    CAST(COD_PLATAFORMA AS STRING)                               AS COD_PLATAFORMA,

    TO_DATE(DAT_CRIACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss')    AS DAT_CRIACAO_REGISTRO_TRANS,
    TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')    AS HR_CRIACAO_REGISTRO_TRANS,
    TO_DATE(DAT_ALTERACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss')                           AS DAT_ALTERACAO_REGISTRO_TRANS,
    TO_CHAR(TO_TIMESTAMP(DAT_ALTERACAO_REGISTRO_TRANS, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')  AS HR_ALTERACAO_REGISTRO_TRANS,
    TO_DATE(DAT_CANCELAMENTO_FAT, 'ddMMMyyyy:HH:mm:ss')         AS DAT_CANCELAMENTO_FAT,
    TO_DATE(DAT_ORIGINAL_VCTO_FAT,'ddMMMyyyy:HH:mm:ss')         AS DAT_ORIGINAL_VCTO_FAT,
    TO_DATE(DAT_ALTERACAO_VCTO_FAT,'ddMMMyyyy:HH:mm:ss')        AS DAT_ALTERACAO_VCTO_FAT,
    TO_DATE(DAT_CRIACAO_FAT,'ddMMMyyyy:HH:mm:ss')               AS DAT_CRIACAO_FAT,
    TO_DATE(DAT_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss')            AS DAT_VENCIMENTO_FAT,
    TO_DATE(DAT_STATUS_FAT,'ddMMMyyyy:HH:mm:ss')                AS DAT_STATUS_FAT,
    TO_CHAR(TO_TIMESTAMP(DAT_STATUS_FAT,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')                AS HR_STATUS_FAT,
    TO_DATE(DAT_MIN_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss')                                 AS DAT_MIN_VENCIMENTO_FAT,
    TO_CHAR(TO_TIMESTAMP(DAT_MIN_VENCIMENTO_FAT,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')        AS HR_MIN_VENCIMENTO_FAT,

    CAST(NUM_BILL_SEQ_FAT AS INT)                                AS NUM_BILL_SEQ_FAT,
    CAST(NUM_SEQ_ACORDO_FAT AS INT)                              AS NUM_SEQ_ACORDO_FAT,

    CAST(IND_ISENCAO_COB_FAT AS STRING)                          AS IND_ISENCAO_COB_FAT,
    CAST(IND_WO AS STRING)                                       AS IND_WO,
    CAST(IND_PDD AS STRING)                                      AS IND_PDD,
    CAST(IND_PCCR AS STRING)                                     AS IND_PCCR,
    CAST(IND_ACA AS STRING)                                      AS IND_ACA,
    CAST(IND_PRIMEIRA_FAT AS STRING)                             AS IND_PRIMEIRA_FAT,
    CAST(IND_FRAUDE AS STRING)                                   AS IND_FRAUDE,

    CAST(VAL_FAT_LIQUIDO AS DECIMAL(18,2))                       AS VAL_FAT_LIQUIDO,
    CAST(VAL_FAT_BRUTO AS DECIMAL(18,2))                         AS VAL_FAT_BRUTO,
    CAST(VAL_FAT_CREDITO AS DECIMAL(18,2))                       AS VAL_FAT_CREDITO,
    CAST(VAL_FAT_AJUSTE AS DECIMAL(18,2))                        AS VAL_FAT_AJUSTE,
    CAST(VAL_FAT_BRUTO_BC AS DECIMAL(18,2))                      AS VAL_FAT_BRUTO_BC,
    CAST(VAL_FAT_PAGAMENTO_BRUTO AS DECIMAL(18,2))               AS VAL_FAT_PAGAMENTO_BRUTO,
    CAST(VAL_FAT_ABERTO AS DECIMAL(18,2))                        AS VAL_FAT_ABERTO,
    CAST(VAL_FAT_ABERTO_LIQ AS DECIMAL(18,2))                    AS VAL_FAT_ABERTO_LIQ,
    CAST(VAL_MULTA_JUROS AS DECIMAL(18,2))                       AS VAL_MULTA_JUROS,
    CAST(VAL_MULTA_CANCELAMENTO AS DECIMAL(18,2))               AS VAL_MULTA_CANCELAMENTO,
    CAST(VAL_PARC_APARELHO_LIQ AS DECIMAL(18,2))                 AS VAL_PARC_APARELHO_LIQ,
    CAST(VAL_FAT_LIQ_JM_MC AS DECIMAL(18,2))                     AS VAL_FAT_LIQ_JM_MC,

    TO_DATE(DAT_ATIVACAO_CONTA_CLI,'ddMMMyyyy:HH:mm:ss')    AS DAT_ATIVACAO_CONTA_CLI,
    TO_CHAR(TO_TIMESTAMP(DAT_ATIVACAO_CONTA_CLI,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')    AS HR_ATIVACAO_CONTA_CLI,
    TO_DATE(DAT_CRIACAO_DW,'ddMMMyyyy:HH:mm:ss')            AS DAT_CRIACAO_DW,
    TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_DW,'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')            AS HR_CRIACAO_DW
FROM raw_base_atraso
""")
df_trusted_atraso.createOrReplaceTempView("trusted_base_atraso")
#df_trusted_atraso.cache()

#print(log(), "Registros Trusted:", df_trusted_atraso.count())
#df_trusted_atraso.printSchema()
df_trusted_atraso.show(5, truncate=False)

+--------------+-----------------+-----------+--------------+----------------------------------------------------------------+------------------+---------+-------------+------------------------+--------------+-------+--------+---------------------+---------+---------------------+---------------------+-------------------+------------------------+-------------------+--------------+--------------------------+-------------------------+----------------------------+---------------------------+--------------------+---------------------+----------------------+---------------+------------------+--------------+-------------+----------------------+---------------------+----------------+------------------+-------------------+------+-------+--------+-------+----------------+----------+---------------+-------------+---------------+--------------+----------------+-----------------------+--------------+------------------+---------------+----------------------+---------------------+-----------------+--

# Join com a dimensão tipo de faturamento

In [10]:
df_trusted_atraso_refined = spark.sql("""
SELECT
    at.*,
    CAST(tf.DW_TIPO_FATURAMENTO AS STRING) AS DW_TIPO_FATURAMENTO,
    CAST(tf.DSC_TIPO_FATURAMENTO AS STRING) AS DSC_TIPO_FATURAMENTO,
    CAST(tf.COD_TIPO_FATURAMENTO AS STRING) AS COD_TIPO_FATURAMENTO,
    TO_DATE(tf.DAT_EXPIRACAO_DW, 'ddMMMyyyy:HH:mm:ss') AS DIM_DAT_EXPIRACAO_DW,                              -- Renaming to avoid conflict
    TO_CHAR(TO_TIMESTAMP(tf.DAT_EXPIRACAO_DW, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_DIM_DAT_EXPIRACAO_DW,  -- New column for time component
    TO_DATE(tf.DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss') AS DIM_DAT_CRIACAO_DW,                                  -- Renaming to avoid conflict
    TO_CHAR(TO_TIMESTAMP(tf.DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_DIM_DAT_CRIACAO_DW,      -- New column for time component
    CAST(tf.DSC_TIPO_FATURAMENTO_ABREV AS STRING) AS DSC_TIPO_FATURAMENTO_ABREV
FROM trusted_base_atraso at
LEFT JOIN df_tipo_faturamento tf
ON at.dw_tipo_faturamento = tf.dw_tipo_faturamento

""")

df_trusted_atraso_refined.createOrReplaceTempView("lake_atraso_refined")
#df_trusted_atraso_refined.cache()

#print(log(), "Registros Trusted:", df_trusted_atraso_refined.count())
#df_trusted_atraso_refined.printSchema()
df_trusted_atraso_refined.show(5)

+--------------+-----------------+-----------+--------------+--------------------+------------------+---------+-------------+------------------------+--------------+-------+--------+---------------------+---------+---------------------+---------------------+-------------------+------------------------+-------------------+--------------+--------------------------+-------------------------+----------------------------+---------------------------+--------------------+---------------------+----------------------+---------------+------------------+--------------+-------------+----------------------+---------------------+----------------+------------------+-------------------+------+-------+--------+-------+----------------+----------+---------------+-------------+---------------+--------------+----------------+-----------------------+--------------+------------------+---------------+----------------------+---------------------+-----------------+----------------------+---------------------+-

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
print("Trusted path:", path_trusted)

df_trusted_atraso_refined.write \
    .partitionBy("ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

# Controle de carga

In [ ]:
df_controle = spark.sql (f"""
SELECT
    '{output_trusted}' AS name_file,
    ts_proc,
    ts_proc_partition,
    count(*) as qtd_registros
from lake_atraso_refined
    GROUP BY 1,2,3
""")

df_controle.show()

# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f'tb_controle_processamento_{bucket_base}_trusted')
print("Control path:", path_control)

df_controle.write \
  .mode('append') \
  .option('compression', 'snappy') \
  .parquet(path_control)